In [39]:
import joblib
import pandas as pd
import os

pipeline = joblib.load("modele_faux_billets_SVM.joblib")

In [40]:
########################################

# Regarder bentoml

#######################################

In [41]:
print(pipeline)

Pipeline(steps=[('imputer',
                 IterativeImputer(estimator=LinearRegression(), random_state=42,
                                  skip_complete=True)),
                ('scaler', StandardScaler()),
                ('model', SVC(C=1, gamma=0.1, probability=True))])


In [42]:
DATA_PATH = "billets.csv"

if not os.path.exists(DATA_PATH):
    print(f"❌ Attention : le fichier '{DATA_PATH}' n'existe pas.")

try:
    df = pd.read_csv(DATA_PATH, sep=";")
    print("✅ Données chargées :", df.shape)
except Exception as e:
    print("❌ Erreur lors du chargement :", e)
    print("⚠️ Tu as bien lu le code avant de l'executer ? 😏")

# On ne garde que les colonnes nécessaires pour la prédiction (diagonal	height_left	height_right	margin_low	margin_up	length)
df = df[["diagonal", "height_left", "height_right", "margin_low", "margin_up", "length"]]

########################################

# Faire les if pour afficher les nan et les imputer

#######################################

df = pipeline['imputer'].transform(df)
df = pipeline["scaler"].transform(df)
#Transformer le tableau numpy en DataFrame pour garder les noms de colonnes
df = pd.DataFrame(df, columns=["diagonal", "height_left", "height_right", "margin_low", "margin_up", "length"])
print("Imputatipon des valeurs manquantes")

prediction = pipeline["model"].predict(df)
# Ajout de la probabilité de true et de false dans le DataFrame
proba = pipeline["model"].predict_proba(df)

# Ajout de la prediction dans le DataFrame (If 1 = Vrai billet, 0 = Faux billet)
df["Prédiction"] = prediction
df["Résultat"] = ["Faux billet" if p == 1 else "Vrai billet" for p in prediction]

# Ajout de la probabilité de true et de false dans le DataFrame
df["Probabilité d'un faux billet"] = (proba[:, 1]*100).round(2)
df["Probabilité d'un vrai billet"] = (proba[:, 0]*100).round(2)

#Display les lignes Vrai billet
display(df[df["Résultat"] == "Faux billet"].head(30))

# Print du nombre de faux et de vrais billets
print(f"Nombre de vrais billets : {(prediction == 0).sum()}")
print(f"Nombre de faux billets : {(prediction == 1).sum()}")


✅ Données chargées : (1500, 7)
Imputatipon des valeurs manquantes


c:\Users\flepineux\AppData\Local\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but SVC was fitted without feature names
  warnings.warn(
c:\Users\flepineux\AppData\Local\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but SVC was fitted without feature names
  warnings.warn(


,diagonal,height_left,height_right,margin_low,margin_up,length,Prédiction,Résultat,Probabilité d'un faux billet,Probabilité d'un vrai billet
591,-0.944983,-0.733798,-0.511499,0.177614,0.643454,-0.564292,1,Faux billet,79.69,20.31
728,-0.056903,0.268164,0.731827,-0.608243,0.858129,-1.043634,1,Faux billet,97.90,2.10
1000,1.061420,-0.266216,-0.045252,0.470384,0.686389,-1.454499,1,Faux billet,100.00,0.00
1001,-0.122687,-0.566804,1.166991,0.747745,-0.086443,-1.580041,1,Faux billet,100.00,0.00
1002,-1.208118,0.368360,1.415656,0.763154,1.373350,-1.659932,1,Faux billet,100.00,0.00
1003,0.206232,1.002936,1.260240,1.102151,0.257038,-0.781138,1,Faux billet,100.00,0.00
1004,1.949500,0.735746,0.949409,1.733919,-0.086443,-1.089286,1,Faux billet,99.64,0.36
1005,-0.254254,0.902740,0.793993,1.333286,0.772259,0.017766,1,Faux billet,99.72,0.28
1006,-1.076550,0.067772,1.011575,0.208432,-0.215248,-2.013732,1,Faux billet,100.00,0.00
1007,-0.418713,0.334962,1.850820,0.716927,0.514648,-1.089286,1,Faux billet,100.00,0.00


Nombre de vrais billets : 1006
Nombre de faux billets : 494


In [43]:
# Print du nombre de lignes du DataFrame avec prediction = 1
print(f"Nombre de lignes avec prediction = 1 : {df[df['Prédiction'] == 1].shape[0]}")

Nombre de lignes avec prediction = 1 : 494


In [44]:
df_verif = pd.read_csv(DATA_PATH, sep=";")

In [45]:
# Merge de df et df_verif sur l'index pour comparer les résultats 
df_merged = df.merge(df_verif['is_genuine'], left_index=True, right_index=True, suffixes=('_pred', '_verif'))
df_merged["is_fake"] = df_merged["is_genuine"].apply(lambda x: 0 if x == 1 else 1)
df_merged = df_merged.drop(columns=["is_genuine"])
df_merged["Résultat correct"] = df_merged.apply(lambda row: "Correct" if row["Prédiction"] == row["is_fake"] else "Incorrect", axis=1)

In [46]:
# Print du nombre de lignes du DataFrame correct et incorrect
print(f"Nombre de lignes correctes : {df_merged[df_merged['Résultat correct'] == 'Correct'].shape[0]}")
print(f"Nombre de lignes incorrectes : {df_merged[df_merged['Résultat correct'] == 'Incorrect'].shape[0]}")

Nombre de lignes correctes : 1490
Nombre de lignes incorrectes : 10


In [47]:
#Affichage des lignes Incorrectes
display(df_merged[df_merged["Résultat correct"] == "Incorrect"].head(30))

,diagonal,height_left,height_right,margin_low,margin_up,length,Prédiction,Résultat,Probabilité d'un faux billet,Probabilité d'un vrai billet,is_fake,Résultat correct
591,-0.944983,-0.733798,-0.511499,0.177614,0.643454,-0.564292,1,Faux billet,79.69,20.31,0,Incorrect
728,-0.056903,0.268164,0.731827,-0.608243,0.858129,-1.043634,1,Faux billet,97.90,2.10,0,Incorrect
1025,0.699610,0.568753,0.638577,-0.947240,0.986934,-0.267557,0,Vrai billet,24.34,75.66,1,Incorrect
1073,0.568042,-1.201381,-0.325000,-0.315473,0.299973,-0.598531,0,Vrai billet,13.15,86.85,1,Incorrect
1083,-0.352930,-1.435172,-0.325000,0.193023,0.257038,-0.199079,0,Vrai billet,9.27,90.73,1,Incorrect
1103,-0.254254,0.067772,-0.542582,-0.099748,0.257038,-0.176253,0,Vrai billet,10.31,89.69,1,Incorrect
1122,0.436475,0.401759,0.762910,-0.500381,1.072804,1.341664,0,Vrai billet,0.08,99.92,1,Incorrect
1160,1.423230,0.067772,1.229157,-0.531198,1.115739,-0.016473,0,Vrai billet,29.85,70.15,1,Incorrect
1407,0.206232,-0.132621,1.104825,-0.392518,-0.730469,-0.290382,0,Vrai billet,1.53,98.47,1,Incorrect
1412,0.962745,-0.299615,-0.760164,-0.515790,0.428778,-0.221905,0,Vrai billet,2.73,97.27,1,Incorrect


In [48]:
display(df_merged.head(30))

,diagonal,height_left,height_right,margin_low,margin_up,length,Prédiction,Résultat,Probabilité d'un faux billet,Probabilité d'un vrai billet,is_fake,Résultat correct
0,-0.484497,2.773070,3.187395,0.069751,-1.116884,0.177547,0,Vrai billet,8.47,91.53,0,Correct
1,-1.635712,-2.236742,-0.822331,-1.085921,-0.687534,0.474282,0,Vrai billet,0.31,99.69,0,Correct
2,2.409986,1.503918,-1.319661,-0.115157,-0.902209,0.554173,0,Vrai billet,0.79,99.21,0,Correct
3,-1.964630,-0.399811,0.047998,-1.317055,-0.601663,0.953625,0,Vrai billet,0.02,99.98,0,Correct
4,-0.747632,0.835943,-1.443994,-0.669879,1.416285,-0.153428,0,Vrai billet,17.74,82.26,0,Correct
5,0.699610,-0.967589,0.483162,-0.084339,-0.859274,0.154721,0,Vrai billet,0.24,99.76,0,Correct
6,1.258771,0.501955,-0.231751,0.162205,0.471713,0.154721,0,Vrai billet,7.52,92.48,0,Correct
7,-0.254254,-0.900792,0.483162,-0.762333,-0.988079,0.462870,0,Vrai billet,0.02,99.98,0,Correct
8,1.686365,-0.366412,-0.791247,-0.731515,0.428778,0.200373,0,Vrai billet,0.42,99.58,0,Correct
9,1.686365,0.134569,0.296663,-0.669879,0.428778,0.885147,0,Vrai billet,0.07,99.93,0,Correct
